## 4.1.2 이미지 파일 로딩
imageio 모듈 사용

In [1]:
import numpy as np
import torch
torch.set_printoptions(edgeitems=2, threshold=50)

In [3]:
import imageio

img_arr = imageio.imread('/content/bobby.jpg')
img_arr.shape

<ipython-input-3-cbc615025980>:3: DeprecationWarning: Starting with ImageIO v3 the behavior of this function will switch to that of iio.v3.imread. To keep the current behavior (and make this warning disappear) use `import imageio.v2 as imageio` or call `imageio.v2.imread` directly.
  img_arr = imageio.imread('/content/bobby.jpg')


(720, 1280, 3)

## 4.1.3 레이아웃 변경하기

In [4]:
img = torch.from_numpy(img_arr) # torch.from_numpy 로 Numpy 사용
out = img.permute(2, 0, 1) # permute로 레이아웃을 변경. stride 사용

out은 img 저장 공간과 동일하며 단순히 텐서 레벨에서 크기와 스트라이드 정보만 변경됐다.

텐서플로는 채널 차원을 마지막에 배치하는 H x W x C 레이아웃이다.(저수준 성능 관점에서 레이아웃은 장단점이 있다)
- 다룰 신경망 입력은 N(배치) x C(채널) x H(높이) x W(너비) 텐서로 저장

In [5]:
batch_size = 3 # 사이즈 크기는 3
batch = torch.zeros(batch_size, 3, 256, 256, dtype=torch.uint8)

In [6]:
import os

data_dir = '/content/image-cats'
filenames = [name for name in os.listdir(data_dir)
             if os.path.splitext(name)[-1] == '.png']
for i, filename in enumerate(filenames):
    img_arr = imageio.imread(os.path.join(data_dir, filename))
    img_t = torch.from_numpy(img_arr)
    img_t = img_t.permute(2, 0, 1)
    img_t = img_t[:3] # <1> 투명도 알파 채널까지 있는 이미지도 있지만, 우리는 RGB만 사용하여 첫 세 개 채널만 유지
    batch[i] = img_t

<ipython-input-6-e73ff2839b3a>:7: DeprecationWarning: Starting with ImageIO v3 the behavior of this function will switch to that of iio.v3.imread. To keep the current behavior (and make this warning disappear) use `import imageio.v2 as imageio` or call `imageio.v2.imread` directly.
  img_arr = imageio.imread(os.path.join(data_dir, filename))


## 4.1.4 데이터 정규화
신경망은 입력값이 대략 0에서 1 사이거나 -1 사이에서 1 사이일 때 훈련 성능이 가장 좋은 특징을 띤다.

따라서 대부분의 경우 텐서를 부동소수점으로 캐스팅하고 픽셀값을 정규화한다.

정규화가 복잡한 이유는 정해놓은 입력 값의 범위를 통해 정규화된 값이 0과 1(혹 -1~1) 사이에 놓여야 하기 때문이다.
- 픽셀 값을 부호 없는 8비트 정수의 최댓값인 255로 나눈다.

In [7]:
batch = batch.float()
batch /= 255.0

In [9]:
# 입력 데이터의 평균과 표준편차를 구해서 평균이 0이고 각 채널값이 표준 편차를 넘지 않게 만들기도 한다.
n_channels = batch.shape[1]
for c in range(n_channels):
    mean = torch.mean(batch[:, c])
    std = torch.std(batch[:, c])
    batch[:, c] = (batch[:, c] - mean) / std  # 표준편차->평균